In [21]:
import torch

torch.cuda.is_available()

True

In [22]:
import torch
import torch.nn as nn
from skrl.models.torch import GaussianMixin, Model
from skrl.utils.spaces.torch import unflatten_tensorized_space
from gymnasium import spaces
import gymnasium as gym
import numpy as np

action_space = spaces.Box(low=-7.5, high=7.5, shape=(2,), dtype=np.float32)
observation_space = spaces.Box(low=-0.0, high=1.0, shape=(3, 64, 64), dtype=np.float32)

class GaussianModel(GaussianMixin, Model):
    def __init__(self, observation_space, action_space, device, clip_actions=False,
                 clip_log_std=True, min_log_std=-20, max_log_std=2, reduction="sum"):
        Model.__init__(self, observation_space, action_space, device)
        GaussianMixin.__init__(self, clip_actions, clip_log_std, min_log_std, max_log_std, reduction)

        # Note: We use Lazy layers, so we must do a dummy pass before loading weights
        self.net_container = nn.Sequential(
            nn.LazyConv2d(out_channels=16, kernel_size=5, stride=2),
            nn.ELU(),
            nn.LazyConv2d(out_channels=32, kernel_size=3, stride=2),
            nn.ELU(),
            nn.LazyConv2d(out_channels=32, kernel_size=3, stride=1),
            nn.ELU(),
            nn.Flatten(),
            nn.LazyLinear(out_features=256),
            nn.ELU(),
            nn.LazyLinear(out_features=self.num_actions),
        )
        self.log_std_parameter = nn.Parameter(torch.full(size=(self.num_actions,), fill_value=0.0), requires_grad=True)

    def compute(self, inputs, role=""):
        # The unflatten utility handles the conversion from a flat tensor to 
        # the (C, H, W) shape expected by your Conv2d layers.
        states = unflatten_tensorized_space(self.observation_space, inputs.get("states"))
        output = self.net_container(states)
        
        return output, self.log_std_parameter, {}


# 2. Instantiate and Load Weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu_device = torch.device("cpu")

# Replace obs_space and act_space with your environment's actual dimensions
policy = GaussianModel(observation_space, action_space, device)

# Load the checkpoint
checkpoint = torch.load('agent_100000.pt', map_location=device)
policy.load_state_dict(checkpoint['policy']) # skrl stores it under 'policy'
policy.eval()

dummy_input = {"states": torch.zeros((1, 3, 64, 64)).to(device)}
policy.compute(dummy_input)

/tmp/ipykernel_262622/1361831091.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('agent_100000.pt', map_location=device)


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [14]:
def get_observation():
    # Replace with your actual observation logic from simulation
    # e.g., [left_speed, right_speed, sensor_data...]
    obs = np.ones([1, 3, 64, 64]) * 255.0
    return torch.from_numpy(obs).to('cuda').unsqueeze(0).float()

try:
    while True:
        obs = get_observation()
        
        # 2. Run Inference
        with torch.no_grad():
            action, _, _ = policy.act({"states": obs}, role="policy")

        # Convert the tensor to a numpy array for the robot motors
        action_np = action.cpu().numpy()[0]
        print("action_np: ", action_np)
        
        # 3. Apply Actions
        robot.set_motors(float(action_np[0]), float(action_np[1]))

except KeyboardInterrupt:
    robot.stop()

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same